### Where to look first: Black–White denial gap by county (MA, 2023)

Counties with ≥50 Black decisioned applications, ranked by gap. **Raw, uncontrolled - signals for review, not proof.**

- **Suffolk County (25025 — Boston): 23.3 pp gap** (Black 44.6% vs White 21.2%), on 1,440 Black applications —the largest reliable gap and the top audit priority.
- Plymouth (25023): 19.5 pp on 1,669 apps —strong evidence.
- Barnstable (25001): 20.8 pp but only 190 apps —larger gap, weaker evidence; needs a confidence interval before prioritizing over Plymouth.
- Gaps range 23.3 → 2.7 pp: the statewide ~17 pp average hides wide county variation. This variation is the actionable signal.

Same caveats apply: no control for income, loan type, or credit score.

In [7]:
# Black-White denial-rate gap BY county where should the auditor look first?
# We restrict to the two high-volume, reliable groups.
subset = decisioned[decisioned["derived_race"].isin(
    ["Black or African American", "White"]
)].copy()

# denial rate per county per race
county = (
    subset.groupby(["county_code", "derived_race"])
    .agg(
        decisioned=("action_taken", "size"),
        denied=("action_taken", lambda s: (s == 3).sum()),
    )
    .reset_index()
)
county["denial_rate"] = county["denied"] / county["decisioned"] * 100

# pivot so Black and White rates sit side by side per county
pivot = county.pivot(index="county_code", columns="derived_race",
                     values=["decisioned", "denial_rate"])
pivot.columns = ["_".join(map(str, c)) for c in pivot.columns]
pivot["gap_pp"] = (pivot["denial_rate_Black or African American"]
                   - pivot["denial_rate_White"]).round(1)

# only keep counties with enough Black applications to be reliable (>=50)
reliable = pivot[pivot["decisioned_Black or African American"] >= 50]
reliable = reliable.sort_values("gap_pp", ascending=False)
reliable[["decisioned_Black or African American",
          "denial_rate_Black or African American",
          "denial_rate_White", "gap_pp"]].round(1)

,decisioned_Black or African American,denial_rate_Black or African American,denial_rate_White,gap_pp
county_code,,,,
25025.0,1440.0,44.6,21.2,23.3
25001.0,190.0,40.5,19.7,20.8
25023.0,1669.0,41.5,22.0,19.5
25009.0,766.0,41.4,22.1,19.2
25017.0,1019.0,38.3,19.4,18.9
25021.0,1097.0,38.0,20.5,17.5
25005.0,1189.0,38.4,22.7,15.7
25013.0,860.0,33.1,21.6,11.6
25015.0,61.0,26.2,16.3,9.9


### First finding: raw denial-rate gap by race (MA, 2023)

**Signal:** Black applicants were denied at 38.3% vs White at 20.9% —a raw gap of ~17 percentage points, on large samples (9,752 vs 98,792 decisioned applications).

**This is a raw, uncontrolled comparison — a signal, not proof.**
- No control for income, loan type, loan amount, or credit score (credit score is absent from public HMDA data entirely).
- Small groups (Native Hawaiian, "2 or more minority races" which is ~292 each) show very high rates but are statistically unreliable; do not lead with them. They need confidence intervals and a minimum-volume threshold before use.
- "Race Not Available" is 26,484 applications (~28% of decisioned) — a large unknown that could shift the picture.

**Correct interpretation:** denial rates differ substantially by race; the Black–White gap is large and well-supported and *warrants further review*. It is not evidence of discrimination, because legitimate factors (especially creditworthiness) cannot be observed in this data.

In [3]:
import pandas as pd
pd.set_option("display.max_columns", 200)

hmda = pd.read_csv("../data/sample/hmda_MA_2023.csv", low_memory=False)

# The denial-rate denominator rule from our metric glossary:
# only decisioned applications (action_taken in 1,2,3). Excludes purchased (6),
# withdrawn (4), incomplete (5), preapprovals (7,8).
decisioned = hmda[hmda["action_taken"].isin([1, 2, 3])].copy()

print("All rows:        ", len(hmda))
print("Decisioned rows: ", len(decisioned))
print("Dropped:         ", len(hmda) - len(decisioned))

All rows:         210643
Decisioned rows:  149649
Dropped:          60994


In [4]:
# Denial rate by race — the first real disparity numbers.
# Denial rate = denied (action=3) / decisioned (action in 1,2,3), per race.
by_race = (
    decisioned
    .groupby("derived_race")
    .agg(
        decisioned=("action_taken", "size"),
        denied=("action_taken", lambda s: (s == 3).sum()),
    )
)
by_race["denial_rate_%"] = (by_race["denied"] / by_race["decisioned"] * 100).round(1)
by_race = by_race.sort_values("denial_rate_%", ascending=False)
by_race

,decisioned,denied,denial_rate_%
derived_race,,,
Native Hawaiian or Other Pacific Islander,292,155,53.1
2 or more minority races,292,145,49.7
Free Form Text Only,109,48,44.0
American Indian or Alaska Native,554,231,41.7
Black or African American,9752,3735,38.3
Race Not Available,26484,6274,23.7
White,98792,20629,20.9
Asian,10238,2121,20.7
Joint,3136,564,18.0
